# Job Market & Skills Analysis

## Research question

**What do analyst, product, and business-intelligence job postings actually ask for, and how do role family, experience level, salary, skills, and industry differ across them?**

This notebook follows the same workflow I use in coursework:

**question → data overview → cleaning → joining → exploratory analysis → visualization → interpretation → limitations**


## 1. Importing


In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (10, 6)


## 2. Data overview

The repository contains multiple related CSV tables rather than one flat file.

- `postings.csv` — one row per job posting
- `job_skills.csv` — bridge table connecting jobs to skill categories
- `skills.csv` — lookup table translating skill codes into names
- `job_industries.csv` — bridge table connecting jobs to industries
- `industries.csv` — industry lookup table
- `companies.csv` and `salaries.csv` — additional company and compensation data

The analysis begins with the posting table, then joins related tables only when a question needs them.


In [ ]:
POSTING_COLUMNS = [
    "job_id",
    "title",
    "description",
    "location",
    "formatted_experience_level",
    "normalized_salary",
    "remote_allowed",
    "formatted_work_type",
]

postings = pd.read_csv("postings.csv", usecols=POSTING_COLUMNS)
job_skills = pd.read_csv("job_skills.csv")
skills = pd.read_csv("skills.csv")
job_industries = pd.read_csv("job_industries.csv")
industries = pd.read_csv("industries.csv")

print("postings:", postings.shape)
print("job_skills:", job_skills.shape)
print("skills:", skills.shape)
print("job_industries:", job_industries.shape)
print("industries:", industries.shape)

postings.head()


# Data cleaning appendix

## 3. Create comparable role families

Raw job titles are too granular for useful comparison. The title classifier groups only clear matches into five role families and leaves ambiguous titles unclassified.

This avoids treating every job with the word “analyst” as a Data Analyst.


In [ ]:
def classify_role(title):
    title = str(title).lower()

    if re.search(r"\b(product manager|associate product manager|product management)\b", title):
        return "Product management"
    if re.search(r"\bproduct analyst\b", title):
        return "Product analyst"
    if re.search(r"\b(business intelligence|bi analyst|business intelligence analyst)\b", title):
        return "Business intelligence"
    if re.search(r"\b(business systems analyst|business system analyst|business analyst)\b", title):
        return "Business analyst"
    if re.search(r"\b(data analyst|analytics analyst|data analytics analyst)\b", title):
        return "Data analyst"

    return pd.NA

postings["role_family"] = postings["title"].apply(classify_role)
target = postings.dropna(subset=["role_family"]).copy()

target["formatted_experience_level"] = (
    target["formatted_experience_level"]
    .fillna("Missing")
    .replace("", "Missing")
)

target["normalized_salary"] = pd.to_numeric(
    target["normalized_salary"],
    errors="coerce",
)
target.loc[target["normalized_salary"] <= 0, "normalized_salary"] = pd.NA

print("All postings:", len(postings))
print("Target-role postings:", len(target))
target[["title", "role_family", "formatted_experience_level", "normalized_salary"]].head()


## 4. Join the relational tables

The skill and industry tables use bridge tables because each job can belong to multiple categories and each category can belong to many jobs.

### Skills join

`job_skills.csv` contains job IDs and abbreviated skill codes. `skills.csv` translates those codes into readable names.

First join those tables on `skill_abr`, then join the result back to the target postings on `job_id`.


In [ ]:
job_skill_names = job_skills.merge(
    skills,
    on="skill_abr",
    how="left",
)

role_skills = target[
    ["job_id", "role_family"]
].merge(
    job_skill_names,
    on="job_id",
    how="left",
)

role_skills.head()


### Industry join

The same logic applies to industries. `job_industries.csv` connects a posting to an `industry_id`; `industries.csv` translates that ID into a readable industry name.


In [ ]:
job_industry_names = job_industries.merge(
    industries,
    on="industry_id",
    how="left",
)

role_industries = target[
    ["job_id", "role_family"]
].merge(
    job_industry_names,
    on="job_id",
    how="left",
)

role_industries.head()


## 5. Extract literal tool mentions from descriptions

The broad skill categories above are useful for relational analysis, but they are not granular enough to distinguish SQL from Tableau or Python.

For those tools, I use a separate keyword scan of the posting description. This is deliberately simple and transparent rather than pretending to be a full NLP model.


In [ ]:
TOOL_PATTERNS = {
    "SQL": r"\bsql\b",
    "Excel": r"\bexcel\b",
    "Python": r"\bpython\b",
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower\s*bi\b",
    "R": r"\bR\b",
    "A/B testing": r"\ba/b test(?:ing)?\b",
    "Jira": r"\bjira\b",
    "Figma": r"\bfigma\b",
}

for tool, pattern in TOOL_PATTERNS.items():
    flags = 0 if tool == "R" else re.IGNORECASE
    target[tool] = (
        target["description"]
        .fillna("")
        .str.contains(pattern, regex=True, flags=flags)
    )


# Exploratory analysis

## 6. Role volume


In [ ]:
role_counts = target["role_family"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(role_counts.index, role_counts.values)
ax.set_title("Target-role postings in the dataset")
ax.set_xlabel("Postings")
ax.set_ylabel("")

for y, value in enumerate(role_counts.values):
    ax.text(value + 1, y, str(value), va="center")

plt.tight_layout()
plt.show()

role_counts


**Why this chart:** ranking is the question here, so a horizontal bar chart is appropriate. This is the one place where the bar chart earns its keep.


## 7. Experience mix by role family


In [ ]:
experience_order = [
    "Internship",
    "Entry level",
    "Associate",
    "Mid-Senior level",
    "Director",
    "Missing",
]

experience_share = (
    pd.crosstab(
        target["role_family"],
        target["formatted_experience_level"],
        normalize="index",
    )
    .reindex(columns=experience_order, fill_value=0)
    .mul(100)
)

fig, ax = plt.subplots(figsize=(11, 6))
left = np.zeros(len(experience_share))

for level in experience_order:
    values = experience_share[level].values
    ax.barh(experience_share.index, values, left=left, label=level)
    left += values

ax.set_xlim(0, 100)
ax.set_xlabel("Share of postings (%)")
ax.set_ylabel("")
ax.set_title("Experience mix differs sharply by role family")
ax.legend(
    title="Experience label",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.tight_layout()
plt.show()

experience_share.round(1)


**Interpretation:** the 100% stacked view makes the seniority mix visible instead of reducing each role to one count. In this sample, Data Analyst is much more early-career-heavy than Product Management or Business Analyst.


## 8. Salary distribution, not just salary averages


In [ ]:
salary_data = target.dropna(subset=["normalized_salary"]).copy()

role_order = (
    salary_data.groupby("role_family")["normalized_salary"]
    .median()
    .sort_values()
    .index
)

salary_groups = [
    salary_data.loc[
        salary_data["role_family"] == role,
        "normalized_salary",
    ].values
    for role in role_order
]

fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    salary_groups,
    vert=False,
    tick_labels=role_order,
    showfliers=True,
)
ax.set_title("Salary distributions vary within each role family")
ax.set_xlabel("Normalized annual salary (USD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

salary_summary = (
    salary_data.groupby("role_family")["normalized_salary"]
    .agg(["count", "median", "mean", "min", "max"])
    .sort_values("median", ascending=False)
)

salary_summary


**Why a box plot:** a single bar for each median hides the spread, outliers, and small sample sizes. The distribution is especially important because the role groups have different seniority mixes.


## 9. Tool mentions by role family


In [ ]:
tool_by_role = (
    target.groupby("role_family")[list(TOOL_PATTERNS)]
    .mean()
    .mul(100)
)

heatmap_data = tool_by_role.drop(
    index="Product analyst",
    errors="ignore",
).T

fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(heatmap_data.values, aspect="auto")

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=25, ha="right")
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Tool mentions by role family")

for row in range(heatmap_data.shape[0]):
    for col in range(heatmap_data.shape[1]):
        ax.text(
            col,
            row,
            f"{heatmap_data.iloc[row, col]:.0f}%",
            ha="center",
            va="center",
        )

fig.colorbar(
    image,
    ax=ax,
    label="Share of postings mentioning tool (%)",
)
plt.tight_layout()
plt.show()

heatmap_data.round(1)


**Interpretation:** the heatmap is more useful than another ranked bar chart because the question is comparative: which tools are characteristic of which roles?


## 10. Broad skill categories from the joined tables


In [ ]:
skill_counts = (
    role_skills.dropna(subset=["skill_name"])
    .groupby(["role_family", "skill_name"])["job_id"]
    .nunique()
    .reset_index(name="postings")
)

top_skill_categories = (
    skill_counts.groupby("skill_name")["postings"]
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

skill_matrix = (
    skill_counts[
        skill_counts["skill_name"].isin(top_skill_categories)
    ]
    .pivot(
        index="skill_name",
        columns="role_family",
        values="postings",
    )
    .fillna(0)
)

skill_matrix


This table is the direct output of the relational join. It lets us compare LinkedIn's broader skill taxonomy with the more granular keyword analysis above.


## 11. Industry concentration from the joined tables


In [ ]:
industry_counts = (
    role_industries.dropna(subset=["industry_name"])
    .groupby("industry_name")["job_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(industry_counts))
ax.hlines(y, 0, industry_counts.values)
ax.plot(industry_counts.values, y, "o")
ax.set_yticks(y)
ax.set_yticklabels(industry_counts.index)
ax.set_xlabel("Unique target-role postings")
ax.set_title("Industries with the most target-role postings")
plt.tight_layout()
plt.show()

industry_counts


**Why a lollipop chart:** this is still a ranking, but visually separating the point from the baseline keeps the notebook from becoming a wall of identical bars.


# Findings

1. **Seniority mix differs substantially by role family.** The stacked experience view makes it clear that Data Analyst is much more early-career-heavy than Product Management or Business Analyst in this sample.

2. **SQL is the strongest recurring technical signal across adjacent roles.** The tool heatmap shows that it appears across analyst, BI, and product postings rather than belonging to only one title family.

3. **Salary cannot be interpreted without seniority and distribution context.** Product Management has the highest observed median, but it also has a more senior experience mix and a wider salary distribution.

4. **The joined tables add a different level of analysis.** Description keywords show concrete tools such as SQL and Tableau; the skill and industry joins reveal broader categories attached to each posting.

5. **Adjacent roles overlap without being interchangeable.** Their experience mix, tool profile, and industry distribution differ enough that one generic “analyst/product” label would hide useful structure.


# Limitations

- This dataset is a snapshot rather than a complete census of the labor market.
- Role classification is rule-based and intentionally conservative.
- Experience labels are missing for some postings.
- Salary coverage is incomplete and normalized salary depends on the source dataset's methodology.
- Literal keyword matching misses synonyms and implied requirements.
- Product Analyst has too few captured postings for stable comparison.
- A job can map to multiple skills and industries, so joined-table counts should be interpreted as associations rather than mutually exclusive categories.
- The analysis is descriptive and does not establish causation.
